# SSL Watermarking in Google Colab

This notebook is a Colab-friendly wrapper for the project in this repository. It installs the dependencies, mounts Google Drive, points to your repo folder, and shows the commands for fitting PCA-whitening, generating keys, marking images, detecting watermarks, and running evaluation.

Before you run the command cells, make sure the repository folder is available in Colab, for example at `/content/ssl` or `/content/drive/MyDrive/ssl`.

## 1) Install dependencies

Run this once per Colab session.

In [ ]:
!pip -q install -r requirements.txt

## 2) Mount Drive and locate the repo

If you uploaded the repository directly into `/content/ssl`, this cell will use it. Otherwise it tries Google Drive.

In [ ]:
from pathlib import Path
import os

from google.colab import drive

drive.mount('/content/drive')

candidate_roots = [
    Path('/content/ssl'),
    Path('/content/drive/MyDrive/ssl'),
    Path('/content/drive/MyDrive/thesis/ssl'),
]
REPO_DIR = next((path for path in candidate_roots if (path / 'src').exists()), None)
if REPO_DIR is None:
    raise FileNotFoundError(
        'Could not find the repo. Put the project folder in /content/ssl or Google Drive, then rerun this cell.'
    )

os.chdir(REPO_DIR)
print(f'Using repo: {REPO_DIR}')
print('Files:', sorted(p.name for p in REPO_DIR.iterdir())[:20])

## 3) Sanity check the runtime

A GPU is strongly recommended for the marking loops.

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 4) Set your data and checkpoint paths

Update these paths to match where your images live in Colab or Drive. The notebook uses the same default filenames as the repo.

In [ ]:
from pathlib import Path

IMAGE_DIR = Path('/content/drive/MyDrive/ssl/images')
EVAL_DIR = Path('/content/drive/MyDrive/ssl/eval')
SAMPLE_IMAGE = Path('/content/drive/MyDrive/ssl/example.png')

CHECKPOINT_DIR = REPO_DIR / 'checkpoints'
OUTPUT_DIR = REPO_DIR / 'outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WHITENING_PATH = CHECKPOINT_DIR / 'whitening.pt'
ZERO_KEY_PATH = CHECKPOINT_DIR / 'key_zero.npy'
MULTI_KEY_PATH = CHECKPOINT_DIR / 'key_multi.npy'

print('IMAGE_DIR =', IMAGE_DIR)
print('EVAL_DIR =', EVAL_DIR)
print('SAMPLE_IMAGE =', SAMPLE_IMAGE)
print('WHITENING_PATH =', WHITENING_PATH)
print('ZERO_KEY_PATH =', ZERO_KEY_PATH)
print('MULTI_KEY_PATH =', MULTI_KEY_PATH)

## 5) Fit PCA-whitening

Use a folder of natural images. This step is required before marking or detection if you do not already have `checkpoints/whitening.pt`.

In [ ]:
!python scripts/fit_pca.py --images-dir "{IMAGE_DIR}" --out "{WHITENING_PATH}" --max-images 1000 --batch-size 16 --image-size 224

## 6) Generate watermark keys

Zero-bit uses one carrier vector. Multi-bit uses `k` orthonormal carriers. The repo defaults to `d=2048`, matching the config files.

In [ ]:
!python -c "from src.keys import save_zerobit_key; save_zerobit_key(r'{ZERO_KEY_PATH}', d=2048)"
!python -c "from src.keys import save_multibit_key; save_multibit_key(r'{MULTI_KEY_PATH}', k=30, d=2048)"

## 7) Mark an image

Pick one of the two modes below. For multi-bit, replace the example message with a bit string of length 30, matching `configs/multi_bit.yaml`.

In [ ]:
ZERO_OUT = OUTPUT_DIR / 'marked_zero.png'
MULTI_OUT = OUTPUT_DIR / 'marked_multi.png'

# Zero-bit
!python scripts/mark.py --mode zero_bit --image "{SAMPLE_IMAGE}" --key "{ZERO_KEY_PATH}" --whitening "{WHITENING_PATH}" --out "{ZERO_OUT}"

# Multi-bit example message: 30 bits
MESSAGE = '010110100111000101011001010011'
!python scripts/mark.py --mode multi_bit --image "{SAMPLE_IMAGE}" --key "{MULTI_KEY_PATH}" --whitening "{WHITENING_PATH}" --message "{MESSAGE}" --out "{MULTI_OUT}"

## 8) Detect or decode

Run detection on the marked image. Zero-bit prints a detection decision and score; multi-bit prints the decoded bit string.

In [ ]:
!python scripts/detect.py --mode zero_bit --image "{ZERO_OUT}" --key "{ZERO_KEY_PATH}" --whitening "{WHITENING_PATH}" --fpr 1e-6
!python scripts/detect.py --mode multi_bit --image "{MULTI_OUT}" --key "{MULTI_KEY_PATH}" --whitening "{WHITENING_PATH}"

## 9) Evaluate a folder

This runs the attack suite over a folder of images and reports TPR for zero-bit or BER/WER for multi-bit.

In [ ]:
!python scripts/evaluate.py --mode zero_bit --images-dir "{EVAL_DIR}" --key "{ZERO_KEY_PATH}" --whitening "{WHITENING_PATH}" --max-images 50
!python scripts/evaluate.py --mode multi_bit --images-dir "{EVAL_DIR}" --key "{MULTI_KEY_PATH}" --whitening "{WHITENING_PATH}" --max-images 50